# Solution F1 · No penalty, L2, L1 and elastic net with few true features and many useless ones

**Type**: Compare

## Objective

Compare four regularizations of the logistic (none, L2, L1, elastic net) on 5 simulated datasets with 100 features of which only **6 informative** and 200 training examples, choosing `C` with
cross-validation **for each model**. Measure log-loss and AUC on 2000 new examples, how many features each model keeps and how many of the six true ones it finds again.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(seed):
    X, y = make_classification(n_samples=2200, n_features=100, n_informative=6, n_redundant=0, class_sep=1.0, flip_y=0.03, shuffle=False, random_state=seed)
    o = np.random.default_rng(seed).permutation(len(y))         # shuffle the rows (the informative columns stay the first 6)
    X, y = X[o], y[o]
    return X[:200], y[:200], X[200:], y[200:]


cv = StratifiedKFold(5, shuffle=True, random_state=0)
Cs = np.logspace(-3, 1, 9)

## Commented solution

In [ ]:
def new_models():
    base = lambda **kw: make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, **kw))
    grid = {"logisticregression__C": Cs}
    search = lambda pipe: GridSearchCV(pipe, grid, cv=cv, scoring="neg_log_loss")
    return {
        "none": make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=20000)),
        "L2": search(base(solver="lbfgs")),
        "L1": search(base(penalty="l1", solver="liblinear")),
        "elasticnet": search(base(penalty="elasticnet", solver="saga", l1_ratio=0.5)),
    }

In [ ]:
rows = []
for seed in range(5):
    X_tr, y_tr, X_te, y_te = generate(seed)
    for name, m in new_models().items():
        m.fit(X_tr, y_tr)
        p = np.clip(m.predict_proba(X_te)[:, 1], 1e-15, 1 - 1e-15)
        w = (m.best_estimator_ if hasattr(m, "best_estimator_") else m)[-1].coef_[0]
        active = np.flatnonzero(np.abs(w) > 1e-8)
        rows.append({"seed": seed, "model": name, "logloss": log_loss(y_te, p), "auc": roc_auc_score(y_te, p),
                     "n_coef": len(active), "n_true": int((active < 6).sum())})
table = pd.DataFrame(rows).groupby("model").mean().drop(columns="seed")
table = table.loc[["none", "L2", "L1", "elasticnet"]]
table.round(3)

In [ ]:
sparse_win = bool(table.loc["L1", "logloss"] < table.loc["L2", "logloss"] and table.loc["elasticnet", "logloss"] < table.loc["L2", "logloss"])
l1_true = table.loc["L1", "n_true"]
print(f"the sparse models beat L2: {sparse_win} | true features found by L1: {l1_true:.1f} out of 6 (with {table.loc['L1', 'n_coef']:.0f} active features in total)")

## Check

In [ ]:
with verify():
    check(set(table.index) == {"none", "L2", "L1", "elasticnet"} and list(table.columns) == ["logloss", "auc", "n_coef", "n_true"], "table with four models and four metrics", "columns: logloss, auc, n_coef, n_true")
    check(table.loc["none", "logloss"] > 3 and table.loc["none", "n_coef"] == 100, "without penalty the test log-loss exceeds 3 and all 100 coefficients are nonzero", "C = 1e6")
    check(table.loc["L2", "n_coef"] == 100 and table.loc["L1", "n_coef"] < 50 and table.loc["elasticnet", "n_coef"] < 60, "L2 zeroes nothing; L1 and elastic net keep fewer than half of the features", "count |coef| > 1e-8")
    check(sparse_win is True and table.loc["L1", "auc"] > table.loc["L2", "auc"] + 0.03, "L1 and elastic net beat L2 on log-loss, and L1 by over 0.03 of AUC", "compare the logloss and auc columns")
    check(table.loc["L2", "logloss"] < table.loc["none", "logloss"] / 5, "L2 reduces the test log-loss by over 5 times compared to no penalty", "compare logloss")
    check(l1_true == table.loc["L1", "n_true"] and l1_true >= 4, "L1 finds on average at least 4 of the 6 informative features", "n_true of the L1 row")
    check(table.loc["L1", "n_coef"] > 12, "but it keeps many useless features (over 12): the selection is not clean", "n_coef of the L1 row")
done()

## What you learned

- **No penalty:** with 200 examples and 100 features the model is separable and overfits completely (test log-loss 5.8, AUC 0.76): it is C1 in a realistic version.
- **L2 shrinks but does not select**: it keeps all the 100 features with small weights and "spreads" the signal over the useless features (log-loss 0.58, AUC 0.78). It works when almost all the features carry a little information.
- **L1 and elastic net win here** (log-loss 0.48, AUC 0.86) because the truth is **sparse**: few features matter and the rest is noise. L1 keeps on average 39 features, not 6: it finds 5.0 of the 6 true ones but
  adds many useless ones. L1's selection is **not reliable** as a variable selection procedure (for that you would need stability and repetition).
- Elastic net (with `l1_ratio=0.5`) does almost as well as L1 and is generally more stable when the features are correlated, but requires the `saga` solver and more time. No rule always holds: if you know the signal is diffuse, L2;
  if it is sparse, L1 or elastic net. `C` and `l1_ratio` are chosen in cross-validation, not by eye.

## Common pitfall

Concluding from a single dataset that "L1 is better than L2" (or vice versa). The answer depends on how the signal is made (sparse or diffuse) and on how many examples you have: here the difference is large because six features out of a hundred really matter. Try
redoing the experiment with 30 informative features out of 100 and the result shrinks or flips.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When would you use L1 instead of L2 for a logistic?</summary>

When I think few variables really matter and I want a sparse, interpretable model, or when I have many more features than examples. L2 when the signal is diffuse or the features are highly correlated (L1 picks one of the group
unstably). Elastic net is the compromise. In all cases I standardize the features and choose `C` (and `l1_ratio`) in cross-validation, remembering that the selection done by L1 must be checked for stability.

</details>